In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 27. Flash Attention and Memory Optimization

> **Learning Goals**
> - Understand how Flash Attention uses the GPU memory hierarchy (SRAM vs HBM)
> - Explain IO complexity and tiling
> - Verify memory savings experimentally

## 27.1 GPU Memory Hierarchy

GPUs have a memory hierarchy:
- **SRAM**: on-chip, very fast, small (~20MB)
- **HBM**: High Bandwidth Memory, usually called GPU memory, around 40-80GB
- **DRAM**: CPU memory, slower, larger

HBM ↔ SRAM transfer is a bottleneck. Standard attention writes and reads the $n \times n$ attention matrix in HBM, which is slow.

## 27.2 Memory Bottleneck of Standard Attention

$$S = QK^\top \in \mathbb{R}^{n \times n}$$  (written to HBM)
$$P = \mathrm{softmax}(S) \in \mathbb{R}^{n \times n}$$  (read/written in HBM)
$$O = PV \in \mathbb{R}^{n \times d}$$  (read/written in HBM)

- HBM access: $O(n^2)$, very large
- SRAM use: almost none

## 27.3 Flash Attention — Tiling and Online Softmax

**Core idea**: do not materialize the $n \times n$ matrix in memory. Process blocks in SRAM.

### Online Softmax
Softmax can be computed incrementally with running $\max$ and $\sum$:
$$m_i^{(j)} = \max(m_i^{(j-1)}, \max_j S_{ij})$$
$$\ell_i^{(j)} = e^{m_i^{(j-1)} - m_i^{(j)}} \ell_i^{(j-1)} + \sum_j e^{S_{ij} - m_i^{(j)}}$$

### Tiling
Move blocks of $Q,K,V$ into SRAM, compute partial attention, and write only the output to HBM.

**IO complexity**:
- Standard: $O(n^2)$ HBM accesses
- Flash: $O(n^2 d / M)$ where $M$ is SRAM size, much smaller

### Recomputation in Backpropagation
During backpropagation, recompute $S,P$ instead of storing them, reducing memory from $O(n^2)$ to $O(n)$.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# standard attention
def standard_attention(Q, K, V):
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-1, -2) / (d_k ** 0.5)  # HBM write
    P = F.softmax(S, dim=-1)  # HBM read/write
    O = P @ V  # HBM read/write
    return O

# Flash Attention approximation (simplified, online softmax)
def flash_attention_naive(Q, K, V, block_size=64):
    """Simplified version of the Flash Attention tiling idea."""
    B, H, N, D = Q.shape
    O = torch.zeros_like(Q)
    # Block Q; use full K/V here. Real Flash blocks both sides.
    for i in range(0, N, block_size):
        Q_block = Q[:, :, i:i+block_size, :]  # (B, H, bs, D)
        S_block = Q_block @ K.transpose(-1, -2) / (D ** 0.5)  # (B, H, bs, N)
        P_block = F.softmax(S_block, dim=-1)
        O[:, :, i:i+block_size, :] = P_block @ V
    return O

# check equivalence
torch.manual_seed(0)
B, H, N, D = 1, 2, 128, 32
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)
V = torch.randn(B, H, N, D)

O_std = standard_attention(Q, K, V)
O_flash = flash_attention_naive(Q, K, V, block_size=32)
print(f"standard vs Flash max error: {(O_std - O_flash).abs().max():.2e}")
print("=> Results are identical. Flash improves memory efficiency.")


## 27.4 PyTorch SDPA Backends

PyTorch `scaled_dot_product_attention` automatically chooses the best backend:
- `flash`: Flash Attention v2 on GPU
- `mem_efficient`: memory-efficient attention on GPU/CPU
- `math`: standard implementation


In [ ]:
# SDPA backend comparison
import time

# check memory effect with a long sequence
def bench_attention(n, d=64, n_heads=8, device='cpu'):
    """Attention Timeand Memory Measurement."""
    Q = torch.randn(1, n_heads, n, d, device=device)
    K = torch.randn(1, n_heads, n, d, device=device)
    V = torch.randn(1, n_heads, n, d, device=device)

    # Standard Implementation
    def std_attn():
        scores = Q @ K.transpose(-1, -2) / (d ** 0.5)
        weights = F.softmax(scores, dim=-1)
        return weights @ V

    # SDPA
    def sdpa_attn():
        return F.scaled_dot_product_attention(Q, K, V)

    # Time Measurement
    for _ in range(2): std_attn()  # warmup
    t0 = time.perf_counter()
    for _ in range(3):
        std_attn()
    t_std = (time.perf_counter() - t0) / 3 * 1000

    for _ in range(2): sdpa_attn()
    t0 = time.perf_counter()
    for _ in range(3):
        sdpa_attn()
    t_sdpa = (time.perf_counter() - t0) / 3 * 1000

    return t_std, t_sdpa

print(f"{'n':>6} | {'Standard (ms)':>15} | {'SDPA (ms)':>12} | {'Speedup':>10}")
print("-" * 55)
for n in [256, 512, 1024, 2048]:
    t_std, t_sdpa = bench_attention(n, device='cpu')
    print(f"{n:>6} | {t_std:>15.3f} | {t_sdpa:>12.3f} | {t_std/t_sdpa:>9.2f}x")


## 27.5 Memory Saving Analysis

Standard attention stores the $n \times n$ attention matrix, so memory is $O(n^2)$.

Flash Attention does not store the attention matrix, so memory is $O(n)$.

Example: $n = 8192, h = 32, d_k = 128, B = 1$:
- Standard: $B \cdot h \cdot n^2 \cdot 4 = 8$ GB (FP32)
- Flash: $O(n)$, roughly 100MB


In [ ]:
# memory analysis
def attention_memory_gb(n, n_heads, d_k, batch=1, dtype_bytes=4):
    """Standard Attentionof Attention Matrix Memory."""
    return batch * n_heads * n * n * dtype_bytes / (1024**3)

print(f"Attention Matrix Memory (FP32):")
print(f"{'n':>6} | {'Memory (GB)':>12} | {'Flash (Estimation)':>14}")
print("-" * 40)
for n in [1024, 2048, 4096, 8192, 16384]:
    mem = attention_memory_gb(n, n_heads=32, d_k=128)
    flash_mem = n * 32 * 128 * 4 / (1024**3)  # roughly O(n)
    print(f"{n:>6} | {mem:>12.3f} | {flash_mem:>14.4f}")
print("\n=> n=8192in Standard 8GB, Flash tens of MB. Difference overwhelming.")


## 27.6 Flash Attention v2/v3

- **v1** (Dao et al., 2022): tiling and online softmax
- **v2** (2023): better workload balancing and optimized backpropagation
- **v3** (2024): optimized for Hopper (H100) with asynchronous execution

## 27.7 Ring Attention — Multi-GPU Distribution

Distribute $Q,K,V$ across GPUs and communicate between GPUs to compute global attention.
- Breaks the memory limit of a single GPU
- Enables 1M+ token contexts

## 27.8 Key Takeaways

| Method | IO Complexity | Memory | Speed |
|---|---|---|---|
| Standard Attention | $O(n^2)$ | $O(n^2)$ | slow |
| Flash Attention v2 | $O(n^2 d/M)$ | $O(n)$ | fast |
| Flash Attention v3 | optimized | $O(n)$ | best on H100 |
| Ring Attention | distributed | $O(n/k)$ (k GPUs) | very long context |

## Exercises

1. Explicitly set PyTorch SDPA backends to `math` and `mem_efficient`, then compare them.
2. Compare standard attention vs SDPA time and memory at sequence lengths 1024, 4096, and 16384.
3. Implement Online Softmax and show that it matches standard softmax.
4. Explain how Flash Attention saves memory during backpropagation.
5. Explain how Ring Attention works across multiple GPUs.

> Solutions: `solutions/ch27_solutions.ipynb`
